In [2]:
pip install keras tensorflow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

data = pd.read_csv("train.txt",sep=";")
data.columns =["Text", "Emotions"]
print(data.head())

                                                Text Emotions
0  i can go from feeling so hopeless to so damned...  sadness
1   im grabbing a minute to post i feel greedy wrong    anger
2  i am ever feeling nostalgic about the fireplac...     love
3                               i am feeling grouchy    anger
4  ive been feeling a little burdened lately wasn...  sadness


In [6]:
texts = data["Text"].tolist()
labels = data['Emotions'].tolist()

# Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

In [ ]:
sequences = tokenizer.texts_to_sequences(texts)
max_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen = max_length)

(15999, 66)


In [11]:
sequences = tokenizer.texts_to_sequences(texts)

In [12]:
padded_sequences

array([[   0,    0,    0, ...,    3,   21, 1254],
       [   0,    0,    0, ...,    2,  494,  437],
       [   0,    0,    0, ...,   29,    5, 3496],
       ...,
       [   0,    0,    0, ...,    3,  101, 1331],
       [   0,    0,    0, ...,  339,    8,   42],
       [   0,    0,    0, ...,   25, 3585,   12]],
      shape=(15999, 66), dtype=int32)

In [13]:
# Encode the string labels to integer
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [14]:
# One-hot encode the labels
one_hot_labels = keras.utils.to_categorical(labels)

In [15]:
one_hot_labels

array([[0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0.]], shape=(15999, 6))

In [16]:
# split the data into training and testing sets
xtrain,xtest,ytrain,ytest = train_test_split(
 padded_sequences,one_hot_labels,test_size=0.2
)

In [17]:
one_hot_labels[0]

array([0., 0., 0., 0., 1., 0.])

In [20]:
# Define the model
model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128, input_length=max_length))

model.add(Flatten())
model.add(Dense(units=128, activation="relu"))
model.add(Dense(units=len(one_hot_labels[0]), activation="softmax"))

model.compile(optimizer ='adam',loss = 'categorical_crossentropy',metrics=['accuracy'])
model.fit(xtrain,ytrain,epochs=10,batch_size=32,validation_data=(xtest,ytest))


c:\Users\modiu\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.4871 - loss: 1.3486 - val_accuracy: 0.6862 - val_loss: 0.9258
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8884 - loss: 0.3591 - val_accuracy: 0.8131 - val_loss: 0.5748
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9840 - loss: 0.0567 - val_accuracy: 0.8216 - val_loss: 0.5905
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9958 - loss: 0.0240 - val_accuracy: 0.8219 - val_loss: 0.6070
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9964 - loss: 0.0164 - val_accuracy: 0.8159 - val_loss: 0.6379
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9970 - loss: 0.0160 - val_accuracy: 0.8141 - val_loss: 0.6920
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9971 - loss: 0.0140 - val_accuracy: 0.8159 - val_loss: 0.6680
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9970 - loss: 0.0128 - val_accuracy: 0.

In [21]:
input_text = "i am ever feeling nostalgic about the fireplace i will know that it is still on the property"

# Preprocess the input text
input_sequence = tokenizer.texts_to_sequences([input_text])
padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length)
prediction = model.predict(padded_input_sequence)
predicted_label = label_encoder.inverse_transform([np.argmax(prediction[0])])
print(predicted_label[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
love
